# Baseline comparison: CPT Qwen vs. SFT Gemma on the SinhalaMMLU test sets

Scores two checkpoints on four test sets:

| set | file | questions |
|---|---|---|
| `mmlu_easy` | `data/SinhalaMMLU/TEST/easy/*.json` | 1851 (1850 scorable: one broken gold label) |
| `shared_easy` | `data/SinhalaMMLU-SharedTask-Test/with_answers/easy.jsonl` | 642 |
| `shared_medium` | `.../medium.jsonl` | 1111 |
| `shared_hard` | `.../hard.jsonl` | 485 |

Models (one per runtime, see step 4):
* `sinhala-mmlu/qwen3.5-4b-si-cpt-easy` - continued pre-training of `Qwen/Qwen3.5-4B-Base`. **Base model: no chat template**, so it gets the official plain-text SinhalaMMLU prompt and the answer is the option digit with the highest next-token logit (the scoring of `scripts/train/probe_eval.py`, so `mmlu_easy` should land near the 0.57 Probe-A result).
* `sinhala-mmlu/gemma-4-e4b-sinhala-mmlu` - reasoning-distilled LoRA merge of `google/gemma-4-E4B-it`, scored with its own chat prompt from the model card. Two modes: **direct** (cell 7, cheap: digit probability right after `Answer: `) and **think** (cell 8, its intended mode: English reasoning, then parse `Answer: N`). The card measures ~51% direct vs ~71% think, so only the think numbers are a fair reading of this model.

Two caveats. The shared-task answers in `with_answers/` are a **proxy key** (~85-90% correct, from `scripts/answer_sharedtask_test.py`), so those three rows are noisy. And the CPT repo is private: your `HF_TOKEN` needs read access to it.

**Runtime -> Change runtime type -> L4 or A100.** A T4 has no bfloat16 and cannot hold the 8B Gemma checkpoint.

## 1. Install

In [2]:
%pip install -q "transformers==5.19.0" accelerate
# If Colab asks to restart the session, restart and continue from cell 2.

## 2. Token and data
Add your Hugging Face token as a Colab secret named `HF_TOKEN` (key icon in the left sidebar, "Notebook access" on).
The upload prompt wants `sinhalammlu_data.zip`, built on your machine with:

```powershell
Compress-Archive -Path "data\SinhalaMMLU","data\SinhalaMMLU-SharedTask-Test" -DestinationPath "$HOME\Downloads\sinhalammlu_data.zip"
```

In [3]:
import shutil
import zipfile
from pathlib import Path

from google.colab import files, userdata
from huggingface_hub import login

login(userdata.get("HF_TOKEN"))

DEST = "/content/data"


def unpack(zip_path, dest=DEST):
    """PowerShell 5.1 writes '\' separators, which Linux keeps inside the file name: flatten them."""
    shutil.rmtree(dest, ignore_errors=True)
    with zipfile.ZipFile(zip_path) as z:
        print("first entries:", z.namelist()[:2])
        for info in z.infolist():
            info.filename = info.filename.replace("\\", "/")
            if not info.filename.endswith("/"):
                z.extract(info, dest)


zips = sorted(Path("/content").glob("*.zip"), key=lambda p: p.stat().st_mtime)
if not zips:
    files.upload()  # pick sinhalammlu_data.zip
    zips = sorted(Path("/content").glob("*.zip"), key=lambda p: p.stat().st_mtime)
unpack(zips[-1])  # the newest zip, whatever Colab renamed it to

DATA = Path(DEST)
for sub in ("SinhalaMMLU/TEST/easy", "SinhalaMMLU-SharedTask-Test/with_answers"):
    print(sub, "->", len(list((DATA / sub).glob("*.json*"))), "files")


Saving sinhalammlu_data.zip to sinhalammlu_data.zip
first entries: ['SinhalaMMLU/TEST/easy/Arts_humanities_easy_f_97.json', 'SinhalaMMLU/TEST/easy/Buddhism_humanities_easy_f_162.json']
SinhalaMMLU/TEST/easy -> 14 files
SinhalaMMLU-SharedTask-Test/with_answers -> 7 files


## 3. Load the four test sets

In [4]:
import json

SHARED = DATA / "SinhalaMMLU-SharedTask-Test" / "with_answers"
SETS = {
    "mmlu_easy": [q for p in sorted((DATA / "SinhalaMMLU" / "TEST" / "easy").glob("*.json"))
                  for q in json.loads(p.read_text(encoding="utf-8"))],
    **{f"shared_{lv}": [json.loads(line) for line in open(SHARED / f"{lv}.jsonl", encoding="utf-8")]
       for lv in ("easy", "medium", "hard")},
}
for name, items in SETS.items():
    print(name, len(items), "questions,", len(items[0]["choices"]), "options")

mmlu_easy 1851 questions, 4 options
shared_easy 642 questions, 4 options
shared_medium 1111 questions, 4 options
shared_hard 485 questions, 4 options


## 4. Pick the model and load it
Run the notebook twice: first with `KEY = "qwen-cpt"`, then **Runtime -> Restart session**, set `KEY = "gemma-sft"`
and run again. One checkpoint at a time keeps the GPU free; cell 9 collects the results of both runs.

In [ ]:
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoModelForImageTextToText, AutoTokenizer

KEY = "qwen-cpt"  # or "gemma-sft"
MODELS = {
    "qwen-cpt": dict(hf_id="sinhala-mmlu/qwen3.5-4b-si-cpt-easy", subfolder="e3",  # e1 / e2 / e3 = epoch copies
                     style="plain", loader=AutoModelForCausalLM),
    "gemma-sft": dict(hf_id="sinhala-mmlu/gemma-4-e4b-sinhala-mmlu", revision="v1.0.0",
                      style="chat", loader=AutoModelForImageTextToText),  # gemma4 is image-text-to-text
}
CFG = MODELS[KEY]
kw = {k: CFG[k] for k in ("revision", "subfolder") if k in CFG}

tokenizer = AutoTokenizer.from_pretrained(CFG["hf_id"], **kw)
model = CFG["loader"].from_pretrained(CFG["hf_id"], dtype=torch.bfloat16, device_map="cuda", **kw).eval()
DIGIT_IDS = [tokenizer.encode(d, add_special_tokens=False)[-1] for d in "1234"]
print(KEY, kw, "| digit token ids", DIGIT_IDS, "->", tokenizer.convert_ids_to_tokens(DIGIT_IDS))


config.json:   0%|          | 0.00/1.98k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.47k [00:00<?, ?B/s]

e3/tokenizer.json: reconstructing file:   0%|          |  0.00B / 20.0MB            

e3/tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/7.76k [00:00<?, ?B/s]

e3/model.safetensors: reconstructing file:   0%|          |  0.00B / 8.41GB            

e3/model.safetensors: downloading bytes:           |  0.00B            

## 5. Prompts
`plain_prompt` is the official SinhalaMMLU prompt verbatim (`scripts/probes/probe_common.py`), ending in
`පිළිතුර: ` **with** the trailing space so the next token is a bare digit. `chat_prompt` is the Gemma card's
prompt from its `scripts/bakeoff_run.py`, including the subject/grade context its validation run used.

In [ ]:
INTRO = "මෙය {} විෂයයට අදාළ බහුවරණ ප්‍රශ්නයකි.\n"
INSTRUCTION = "පහත ප්‍රශ්නයට 1, 2, 3, 4 යන පිළිතුරුවලින් නිවැරදි හෝ ඉතාමත් ගැළපෙන පිළිතුර තෝරන්න.\n\n"
SYSTEM = ("You are an expert teacher of the Sri Lankan national school curriculum (grades 6-13), fluent in "
          "Sinhala. You will receive a multiple-choice question written in Sinhala with four numbered options. "
          "Exactly one option is correct. Read the Sinhala carefully, including negations such as "
          "'නොවන' / 'නොවේ' (NOT). You may reason in English. "
          "Finish with the final answer in the form 'Answer: <number>'.")


def options(q):
    return "\n".join(f"{i}. {c.strip()}" for i, c in enumerate(q["choices"], start=1))


def plain_prompt(q):
    subject = q["metadata"].get("subject_original") or q["subject"]
    return f"{INTRO.format(subject)}{INSTRUCTION}ප්‍රශ්නය: {q['question']}\n{options(q)}\nපිළිතුර: "


def chat_prompt(q, think=False):
    user = (f"විෂය (Subject): {q['subject']} | ශ්‍රේණිය (Grade): {q['metadata'].get('grade')}\n\n"
            f"ප්‍රශ්නය (Question):\n{q['question']}\n\nවිකල්ප (Options):\n{options(q)}\n\n"
            "Which option (1, 2, 3 or 4) is correct? End your reply with 'Answer: <number>'.")
    text = tokenizer.apply_chat_template(
        [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}],
        tokenize=False, add_generation_prompt=True, enable_thinking=think)
    return text if think else text + "Answer: "  # prime the answer, then read the digit


build = plain_prompt if CFG["style"] == "plain" else chat_prompt
print(build(SETS["shared_easy"][0]))

## 6. Scoring helpers

In [ ]:
import inspect
import time

KEEP = {"logits_to_keep": 1} if "logits_to_keep" in inspect.signature(model.forward).parameters else {}


@torch.inference_mode()
def predict(prompt, n_options=4):
    """The option digit with the highest logit as the next token (index 0 = option 1)."""
    ids = tokenizer(prompt, add_special_tokens=False, return_tensors="pt").to(model.device)
    logits = model(**ids, **KEEP).logits[0, -1]  # KEEP: keep only the last position when supported
    return int(logits[DIGIT_IDS[:n_options]].argmax()) + 1


def score(name, items, preds, tag):
    """Accuracy over the items with a usable gold label; appends to /content/results_<tag>.json."""
    hits = [p == q["answer"] for q, p in zip(items, preds) if 1 <= q["answer"] <= len(q["choices"])]
    out = Path(f"/content/results_{tag}.json")
    results = json.loads(out.read_text()) if out.exists() else {}
    results[name] = {"accuracy": sum(hits) / len(hits), "n": len(hits)}
    out.write_text(json.dumps(results, indent=1))
    Path(f"/content/preds_{tag}_{name}.jsonl").write_text("".join(
        json.dumps({"q_no": q.get("q_no"), "subject": q.get("subject"), "gold": q["answer"], "pred": p},
                   ensure_ascii=False) + "\n" for q, p in zip(items, preds)), encoding="utf-8")
    print(f"{name}: {results[name]['accuracy']:.2%} (n={len(hits)})")

## 7. Direct scoring, all four sets
One forward pass per question: a few minutes per set. For the CPT model this is the official evaluation method;
for the Gemma model it is the card's "direct" mode, which leaves most of its accuracy unused - cell 8 is the one
to trust for that checkpoint.

In [ ]:
for name, items in SETS.items():
    t0 = time.time()
    preds = [predict(build(q), len(q["choices"])) for q in tqdm(items, desc=name)]
    score(name, items, preds, KEY)
    print(f"  {time.time() - t0:.0f}s")

## 8. Gemma only: reasoning mode (needs `KEY = "gemma-sft"`)
The checkpoint's intended inference: thinking on, up to `MAX_NEW` tokens of English reasoning, then `Answer: N`
parsed out of the reply; an unparseable reply falls back to the direct digit score. Expect roughly 1-3 s per
question on an A100, so the defaults cover an evenly spread subset of each set - raise `LIMIT`, or set it to
`None`, if you have the GPU hours for all 4089 questions.

In [ ]:
import re

ANSWER_RE = re.compile(r"(?:Answer|පිළිතුර)\s*[:：]?\s*\**\s*\(?([1-4])\)?")
THINK_END = "<channel|>"  # end of Gemma 4's thinking channel, from the card's bakeoff_run.py
LIMIT, BATCH, MAX_NEW = 200, 8, 2048
tokenizer.padding_side = "left"


@torch.inference_mode()
def think_preds(items):
    out = []
    for i in tqdm(range(0, len(items), BATCH)):
        chunk = items[i:i + BATCH]
        ids = tokenizer([chat_prompt(q, think=True) for q in chunk], add_special_tokens=False,
                        padding=True, return_tensors="pt").to(model.device)
        gen = model.generate(**ids, max_new_tokens=MAX_NEW, do_sample=False)
        texts = tokenizer.batch_decode(gen[:, ids["input_ids"].shape[-1]:], skip_special_tokens=False)
        for q, text in zip(chunk, texts):
            reply = text.split(THINK_END, 1)[-1]  # the answer after the thinking block, else the whole reply
            m = ANSWER_RE.findall(reply) or ANSWER_RE.findall(text)
            out.append(int(m[-1]) if m else predict(chat_prompt(q), len(q["choices"])))  # fall back to direct
    return out


for name, items in SETS.items():
    subset = items if not LIMIT else items[::max(1, len(items) // LIMIT)][:LIMIT]  # spread over all subjects
    t0 = time.time()
    score(name, subset, think_preds(subset), KEY + "-think")
    print(f"  {time.time() - t0:.0f}s for {len(subset)} of {len(items)}")

## 9. Comparison table
Reads every `results_*.json` in this runtime, so run it after the last model. Accuracy in %; `qwen-cpt` and
`gemma-sft` are direct scoring, `gemma-sft-think` is the reasoning mode.

In [ ]:
import glob

import pandas as pd

runs = {Path(p).stem[len("results_"):]: json.loads(Path(p).read_text())
        for p in sorted(glob.glob("/content/results_*.json"))}
df = pd.DataFrame({k: {s: v["accuracy"] * 100 for s, v in r.items()} for k, r in runs.items()})
df = df.reindex(["mmlu_easy", "shared_easy", "shared_medium", "shared_hard"])
df.loc["shared_macro"] = df.loc[["shared_easy", "shared_medium", "shared_hard"]].mean()
print(df.round(2).to_string())
print("\nquestions scored:", {k: {s: v["n"] for s, v in r.items()} for k, r in runs.items()})
df.round(2).to_csv("/content/comparison_baseline.csv")
files.download("/content/comparison_baseline.csv")